In [ ]:
"""

Real World Workflow

1. Data Understanding
2. EDA
3. Feature Selection
4. Scaling
5. Dendrogram Analysis
6. Baseline Hierarchical Clustering
7. Cluster Assignment
8. Internal Validation
9. Linkage Comparison
10. Cluster Count Tuning
11. PCA Visualization
12. Cluster Profiling
13. Stability Analysis
14. Deployment Readiness
15. Interview Questions

==================================================================
"""

# ==========================================================
# STEP 0 : IMPORTS
# ==========================================================

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import (silhouette_score,davies_bouldin_score,calinski_harabasz_score)
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import (dendrogram,linkage)
import joblib


In [ ]:
# ==========================================================
# STEP 1 : DATA UNDERSTANDING
# ==========================================================

wine = load_wine()
df = pd.DataFrame(wine.data,columns=wine.feature_names)

print(df.head())
print(df.info())
print(df.describe())
print(df.shape)

In [ ]:
# ==========================================================
# STEP 2 : EDA
# ==========================================================

print(df.isnull().sum())
print(df.duplicated().sum())
plt.figure(figsize=(12,8))
sns.heatmap(df.corr(),cmap='coolwarm')
plt.title("Correlation Matrix")
plt.show()

In [ ]:
# ==========================================================
# STEP 3 : FEATURE SELECTION
# ==========================================================

X = df.copy()

In [ ]:
# ==========================================================
# STEP 4 : SCALING
# ==========================================================

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# ==========================================================
# STEP 5 : DENDROGRAM ANALYSIS
# ==========================================================

"""
Used to determine possible cluster count.
"""

plt.figure(figsize=(14,8))
linked = linkage(X_scaled,method='ward')

dendrogram(linked)
plt.title("Hierarchical Dendrogram")
plt.xlabel("Samples")
plt.ylabel("Distance")
plt.show()

In [ ]:
# ==========================================================
# STEP 6 : BASELINE MODEL
# ==========================================================

hc = AgglomerativeClustering(n_clusters=3,linkage='ward')

In [ ]:
# ==========================================================
# STEP 7 : CLUSTER ASSIGNMENT
# ==========================================================

labels = hc.fit_predict(X_scaled)
df["cluster"] = labels
print(df["cluster"].value_counts())

In [ ]:
# ==========================================================
# STEP 8 : INTERNAL VALIDATION
# ==========================================================

sil = silhouette_score(X_scaled,labels)
db = davies_bouldin_score(X_scaled,labels)
ch = calinski_harabasz_score(X_scaled,labels)

print("Silhouette:",sil)
print("Davies Bouldin:",db)
print("Calinski Harabasz:",ch)

In [ ]:
# ==========================================================
# STEP 9 : LINKAGE COMPARISON
# ==========================================================

linkages = ["ward","complete","average","single"]
results = []

for method in linkages:
    model = AgglomerativeClustering(n_clusters=3,linkage=method)
    lbl = model.fit_predict(X_scaled)
    score = silhouette_score(X_scaled,lbl)
    results.append([method,score])

comparison = pd.DataFrame(results,columns=["Linkage","Silhouette"])
print(comparison)

In [ ]:
# ==========================================================
# STEP 10 : CLUSTER COUNT TUNING
# ==========================================================

scores = []

for k in range(2,11):

    model = AgglomerativeClustering(n_clusters=k,linkage='ward')
    lbl = model.fit_predict(X_scaled)
    score = silhouette_score(X_scaled,lbl)
    scores.append(score)

plt.figure(figsize=(8,5))
plt.plot(range(2,11),scores,marker='o')
plt.xlabel("Clusters")
plt.ylabel("Silhouette")
plt.title("Cluster Tuning")
plt.show()
best_k = np.argmax(scores)+2
print("Best K:",best_k)

In [ ]:
# ==========================================================
# STEP 11 : PCA VISUALIZATION
# ==========================================================

best_model = AgglomerativeClustering(n_clusters=best_k,linkage='ward')

labels = best_model.fit_predict(X_scaled)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10,6))
scatter = plt.scatter(X_pca[:,0],X_pca[:,1],c=labels,cmap='viridis')

plt.colorbar(scatter)
plt.title("Hierarchical Clusters")
plt.show()

In [ ]:
# ==========================================================
# STEP 12 : CLUSTER PROFILING
# ==========================================================

df["cluster"] = labels
profile = df.groupby("cluster").mean()
print(profile)
print(df["cluster"].value_counts())

In [ ]:
# ==========================================================
# STEP 13 : STABILITY ANALYSIS
# ==========================================================

for method in ["ward","complete","average"]:
    model = AgglomerativeClustering(n_clusters=best_k,linkage=method)
    lbl = model.fit_predict(X_scaled)
    score = silhouette_score(X_scaled,lbl)
    print(method,score)

In [ ]:
# ==========================================================
# STEP 14 : DEPLOYMENT READINESS
# ==========================================================

"""
AgglomerativeClustering does not support predict() like KMeans.

Production approach:
1. Retrain periodically
or
2. Use KMeans after
   discovering K.
Very common industry practice.
"""

joblib.dump(scaler,"hierarchical_scaler.pkl")
print("Scaler Saved")

# ==========================================================
# STEP 15 : INTERVIEW QUESTIONS
# ==========================================================

"""
1. What is Hierarchical Clustering?
2. Difference between
   Agglomerative and Divisive?
3. What is a dendrogram?
4. How is K selected?
5. What is linkage?
6. Single linkage?
7. Coplete linkage?
8. Average linkage?
9. Ward linkage?
10. Why Ward is popular?
11. Difference from KMeans?
12. Advantages over KMeans?
13. Disadvantages?
14. Why scaling is needed?
15. What is cluster profiling?
16. How is clustering validated?
17. What is silhouette score?
18. What is chaining effect?
19. Why hierarchical clustering
    is expensive?
20. Time complexity?
21. When would you use
    Hierarchical instead of KMeans?
22. Can hierarchical clustering
    predict new samples?
23. What is cophenetic distance?
24. What are real-world use cases?
25. Explain end-to-end workflow.
"""